# Test whether KRAS mutation type and KRAS expression level affect how strongly cells lose viability after KRAS knockout.


In [ ]:
import pandas as pd 

KRAS_data = pd.read_csv("KRAS_data.csv")
print(KRAS_data.shape)
KRAS_data

In [ ]:
# Two ModelIDs originate from the same patient.
KRAS_data[KRAS_data['PatientID'].duplicated(keep=False)].sort_values('PatientID')

In [ ]:
cond_1 = (KRAS_data["KRAS_hotspot"] == 1)

display("with hotspot:", KRAS_data.loc[cond_1]['KRAS_GeneEffect'].describe())
display("without hotspot:", KRAS_data.loc[~cond_1]['KRAS_GeneEffect'].describe())

In [ ]:
with_hotspot = KRAS_data.loc[
    KRAS_data["KRAS_hotspot"].eq(1),
    "KRAS_GeneEffect"
].dropna()
print(len(with_hotspot))

without_hotspot = KRAS_data.loc[
    KRAS_data["KRAS_hotspot"].eq(0),
    "KRAS_GeneEffect"
].dropna()
print(len(without_hotspot))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
fig, ax = plt.subplots(figsize=(8, 5))
arrays = [without_hotspot.to_numpy(), with_hotspot.to_numpy()]
ax.boxplot(arrays, positions=[0, 1], widths=0.4, showfliers=False)
for position, values in enumerate(arrays):
    ax.scatter(position + rng.uniform(-0.12, 0.12, len(values)), values,
               alpha=0.75, s=35)
ax.set_xticks([0, 1], ["Without hotspot", "With hotspot"])
ax.set_ylabel("KRAS Gene Effect (lower means a stronger knockout effect)")
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.set_title("KRAS knockout effect in hotspot and non-hotspot cell lines")
fig.tight_layout()
plt.show()

In [ ]:
from scipy import stats

result = stats.ttest_ind(
    with_hotspot,
    without_hotspot,
    equal_var=False
)

print("t-statistic:", result.statistic)
print(f"p-value: {result.pvalue:.12f}")

In the analyzed cell lines, the mean KRAS knockout effect is more negative in the KRAS hotspot group than in the non-hotspot group. Welch’s test supports the statistical significance of this difference.


# Is KRAS expression associated with its knockout effect?


In [ ]:
KRAS_data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for hotspot, color, label in [
    (0, "steelblue", "Without hotspot"),
    (1, "tomato", "With hotspot"),
]:
    group = KRAS_data.loc[KRAS_data["KRAS_hotspot"].eq(hotspot)]

    ax.scatter(
        group["KRAS_expression"],
        group["KRAS_GeneEffect"],
        color=color,
        label=label,
        alpha=0.8,
        s=50,
    )

ax.set_xlabel("KRAS expression, log₂(TPM + 1)")
ax.set_ylabel("KRAS Gene Effect")
ax.set_title("KRAS expression and its knockout effect")
ax.legend(title="KRAS status")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


from scipy.stats import spearmanr

for hotspot in [0, 1]:
    group = KRAS_data.loc[
        KRAS_data["KRAS_hotspot"].eq(hotspot),
        ["KRAS_expression", "KRAS_GeneEffect"]
    ].dropna()

    rho, p = spearmanr(
        group["KRAS_expression"],
        group["KRAS_GeneEffect"]
    )

    print(
        f"Hotspot={hotspot}: n={len(group)}, "
        f"correlation={rho:.3f}, p-value={p:.4f}"
    )

The plot suggests that RNA expression level is not associated with the effect of KRAS knockout.


# Compare two predictions on held-out cell lines: hotspot only versus hotspot plus expression.


In [ ]:
KRAS_data.head()

In [ ]:
import numpy as np
from pprint import pprint

from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Compare all methods on the same rows
data = KRAS_data.dropna(
    subset=["KRAS_hotspot", "KRAS_expression", "KRAS_GeneEffect"]
).reset_index(drop=True)

y = data["KRAS_GeneEffect"]

# If PatientID is unknown, temporarily treat the model as its own group
groups = data["PatientID"].fillna(data["ModelID"])

# Use identical splits for all methods
cv = GroupKFold(n_splits=5)
splits = list(cv.split(data, y, groups=groups))
pprint(splits)

In [ ]:
methods = {
    "Overall mean": (
        DummyRegressor(strategy="mean"),
        ["KRAS_hotspot"],
    ),
    "Hotspot only": (
        LinearRegression(),
        ["KRAS_hotspot"],
    ),
    "Hotspot + expression": (
        LinearRegression(),
        ["KRAS_hotspot", "KRAS_expression"],
    ),
}

predictions = {}
results = []

for name, (model, columns) in methods.items():
    prediction = cross_val_predict(
        model,
        data[columns],
        y,
        cv=splits,
    )

    predictions[name] = prediction

    results.append({
        "Method": name,
        "MAE": mean_absolute_error(y, prediction),
        "RMSE": np.sqrt(mean_squared_error(y, prediction)),
    })

results = pd.DataFrame(results).set_index("Method")
display(results)

# KRAS_data_EXPANDED

In [ ]:
import pandas as pd
KRAS_data_exp = pd.read_csv("/Users/nident/Desktop/JOB/Cancer Cell Line Encyclopedia (CCLE)/data/pilot/KRAS_data_expanded.csv")
print(KRAS_data_exp.shape)
KRAS_data_exp.head()